## Why do we emulate?

Complex simulations can be too **slow** and **expensive** for ...

:::: {.columns}

::: {.column width="50%"}
* (Fast) prediction
* Sensitivity analysis
* Optimization
* Uncertainty Quantification
:::

::: {.column width="50%"}
![](pics/earth_simulation.png)
:::

::: footer
Intro
:::


::::

## Building an emulator is hard

1) Experimental design
    * If each run is expensive, which data should we evaluate the simulation at?

2) Creating the emulator
    * Which model? Gaussian process, neural network, ...
    * Which hyperparameters? 

* The goal of `autoemulate` is to make this easy! 
    * currently focusing on 2.

::: footer
Intro
:::

## Emulating a rocket simulation with `autoemulate`

![](pics/rocket2.png){fig-align="center"}

::: footer
Rocket simulation
:::

## Rocket simulation
:::: {layout="[ 60, 40 ]"}

::: {#first-column}


In [ ]:
from rocket import rocket_simulator
from autoemulate.experimental_design import LatinHypercube
import matplotlib.pyplot as plt
import numpy as np
# Example usage with LatinHypercube and the simulator
lhd = LatinHypercube([(0., 90.), (1000., 10000.)])
X = lhd.sample(50)
y = np.array([rocket_simulator(x) for x in X])

# Extract launch angles and thrusts for plotting
launch_angles = X[:, 0]
thrusts = X[:, 1]

# Create the scatter plot
plt.figure(figsize=(7, 5))
sc = plt.scatter(launch_angles, thrusts, c=y, cmap='plasma', s=50)  # Increase point size to 50
cbar = plt.colorbar(sc)  
cbar.set_label('Maximum Altitude (m)', fontsize=15)
plt.xlabel('Launch Angle (degrees)', fontsize=15)
plt.ylabel('Thrust (Newtons)', fontsize=15)
plt.grid(True)
plt.show()

:::

::: {#second-column}

* Input X: launch angle, thrust  
* Output y: maximum altitude 
* Problem: simulation takes too long 
* Solution: build an emulator

:::

::::

::: footer
Rocket simulation
:::

## `autoemulate` - building a rocket emulator


In [ ]:
#| echo: True
#| output: asis
from autoemulate.compare import AutoEmulate

ae = AutoEmulate()
ae.setup(X, y)               # X=thrust,launch angle, y=max altitude
best_emulator = ae.compare() # compares emulators

. . .

Behind the scenes, `autoemulate`

* fits and cross-validates various models
* optimises hyperparameters
* returns the best model 


## {.scrollable .smaller}


In [ ]:
#| echo: True
ae.print_results()

:::


## 


In [ ]:
#| echo: True
#| layout-nrow: 2
ae.plot_results()

:::

## 


In [ ]:
#| echo: True
#| layout-nrow: 2
ae.plot_model(best_emulator, figsize=(4, 3))

## `autoemulate` - using the rocket emulator


In [ ]:
#| echo: True
#| output: asis
# Create a grid of launch angles and thrusts
X = np.array(np.meshgrid(np.linspace(0, 90, 100), np.linspace(1000, 10000, 100))).T.reshape(-1, 2)
# Emulate the rocket simulation using the best model
y = best_emulator.predict(X)

::: {.fragment}


In [ ]:
#| fig-align: center
launch_angles = X[:, 0]
thrusts = X[:, 1]

# Create the scatter plot
plt.figure(figsize=(5, 4))
sc = plt.scatter(launch_angles, thrusts, c=y, cmap='plasma', s=50)  # Increase point size to 50
cbar = plt.colorbar(sc)  
cbar.set_label('Maximum Altitude (m)', fontsize=15)
plt.xlabel('Launch Angle (degrees)', fontsize=15)
plt.ylabel('Thrust (Newtons)', fontsize=15)
plt.grid(True)
plt.show()

:::

## `autoemulate` - customisation

-> idea: balance simplicity and customisation

In [ ]:
#| echo: true
#| eval: false
#| code-line-numbers: "1|4|5|6|7|8"
# set up
ae = AutoEmulate()
ae.setup(X, y,                      # simulation input and output
         use_grid_search=True,      # search for best hyperparameters
         grid_search_type='random', # hyperparameter search strategy
         normalise=True,            # standardise input 
         fold_strategy='kfold',     # use k-fold cross-validation
         n_jobs=4)                  # parallelise computation
# ae.compare()

## Summary

* `autoemulate` **makes emulation easy**
    * takes simulation input and output
    * returns emulator model
* current features
    * model selection 
    * hyperparameter optimisation
    * parallelisation
    * compatible with the [scikit-learn](https://scikit-learn.org/stable/){preview-link="true"} ecosystem


## (Near-)future work

* deep learning + GPU support
* visual diagnostics for each model
* make contribution easy (to allow new models, metrics etc.)
* automating the *experimental design* step
* keep it simple

## Collaboration

* searching for new datasets/simulations
* ideas for models / features
* feedback on `autoemulate` once the MVP is ready
* and of course: [code](https://github.com/alan-turing-institute/autoemulate){preview-link="true"}

## Thanks to:

Collaborators: Steven Niederer, Eric Daub

And a lot of people for discussions / data:

::: {.columns }
::: {.column width="50%" .nonincremental}
* Imperial:
    * Marina Strocchi
* BAS:
    * Rosie Williams
    * Ieva Kazlauskaite
    * Robert Arthern
:::

::: {.column width="50%" .nonincremental}
* DT-TRIC / Turing:
    * Kalle Westerling
    * Sophie Arana
    * Keith Worden
    * Zack Xuereb Conti
:::
:::



## `autoemulate` GitHub repo

<https://github.com/alan-turing-institute/autoemulate>

## Glossary 1
* **Emulator**: also metamodel, surrogate model. A model that approximates a simulation, but runs faster and more efficient. Could be any machine learning model. [Gaussian Processes](https://en.wikipedia.org/wiki/Gaussian_process_emulator) are a popular choice.
* **Experimental design**: choosing which points to run the simulation at to create a dataset for training an emulator. A common technique is [Latin Hypercube Sampling](https://en.wikipedia.org/wiki/Latin_hypercube_sampling).

## Glossary 2
* **Hyperparameters**: parameters of a model that are not learned from the data, such as the kernel type in a Gaussian Process or the number of hidden layers in a neural network.
* **Cross-validation**: A method to evaluate models. The idea is to split data into a training and test set, train the model on the training set and evaluate it on the test set. This is repeated multiple times with different splits of the data to get a more robust estimate of the model performance.